In [ ]:
# =============================================================================
# COMPLETE STUDENT HABITS ANALYSIS
# Based on WDBC2026_1.R, WDBC2026_2.R, WDBC2026_3.R, WDBC2026_4_1.R, 
# WDBC2026_4_2.R, and IdealClustering.R
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, OPTICS
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from scipy.spatial.distance import pdist
import umap
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0-8-whitegrid')
sns.set_context("notebook", font_scale=1.2)

# =============================================================================
# 1. LOAD DATA & CREATE QUANTILE LABELS
# =============================================================================

# Load data
df = pd.read_csv("habits.csv")
print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

# Create quantile groups (like ntile() in R)
df['exam_quantile'] = pd.qcut(df['exam_score'], q=4, 
                               labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"])

print("\nQuantile counts:")
print(df['exam_quantile'].value_counts().sort_index())

# Display quantile boundaries
quantile_boundaries = pd.qcut(df['exam_score'], q=4, retbins=True)[1]
print(f"\nQuantile boundaries: {quantile_boundaries}")

# =============================================================================
# 2. EDA & PREPROCESSING (like WDBC2026_1.R)
# =============================================================================

# Identify numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns

# 2.1 Histograms
df[numeric_cols].hist(figsize=(15, 12), bins=20, color='darkmagenta', edgecolor='black')
plt.suptitle("Histograms of Numeric Variables", fontsize=16)
plt.tight_layout()
plt.show()

# 2.2 Boxplots by quantile
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x='exam_quantile', y=col, ax=axes[i], palette='viridis')
    axes[i].set_title(col)
    axes[i].set_xlabel('')
plt.tight_layout()
plt.show()

# 2.3 Density plots by quantile
fig, axes = plt.subplots(3, 4, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    for quantile in df['exam_quantile'].unique():
        subset = df[df['exam_quantile'] == quantile][col]
        sns.kdeplot(subset, ax=axes[i], label=quantile, alpha=0.5)
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

# 2.4 Correlation matrix
corr_matrix = df[numeric_cols].corr(method='spearman')
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title("Spearman Correlation Matrix")
plt.show()

# 2.5 Pairwise scatterplots (sample for clarity)
sample_df = df.sample(min(200, len(df)), random_state=42)
sns.pairplot(sample_df, hue='exam_quantile', 
             vars=['study_hours_per_day', 'social_media_hours', 
                   'attendance_percentage', 'sleep_hours', 'exam_score'])
plt.suptitle("Pairwise Relationships by Exam Quantile", y=1.02)
plt.show()

# =============================================================================
# 3. PREPARE DATA FOR MODELING
# =============================================================================

# Encode categorical variables
categorical_cols = ['gender', 'part_time_job', 'diet_quality', 
                    'parental_education_level', 'internet_quality', 
                    'extracurricular_participation']

# One-hot encode
X = pd.get_dummies(df.drop(['exam_quantile', 'exam_score'], axis=1), 
                   columns=categorical_cols, drop_first=True)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nFeatures after encoding: {X.shape[1]}")

# =============================================================================
# 4. PCA (like WDBC2026_1.R)
# =============================================================================

# 4.1 Fit PCA
pca = PCA()
pca_result = pca.fit_transform(X_scaled)

# Explained variance
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# 4.2 Scree plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(range(1, len(explained_var)+1), explained_var, color='#B53389')
ax1.plot(range(1, len(explained_var)+1), explained_var, 'o-', color='darkblue')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('% Variance')
ax1.set_title('Scree Plot')

ax2.bar(range(1, len(cumulative_var)+1), cumulative_var, color='#F25E52')
ax2.plot(range(1, len(cumulative_var)+1), cumulative_var, 'o-', color='darkblue')
ax2.set_xlabel('Principal Component')
ax2.set_ylabel('Cumulative % Variance')
ax2.set_title('Cumulative Scree Plot')
plt.tight_layout()
plt.show()

print(f"\nVariance explained by first 2 PCs: {cumulative_var[1]:.2%}")
print(f"Variance explained by first 3 PCs: {cumulative_var[2]:.2%}")
print(f"Variance explained by first 4 PCs: {cumulative_var[3]:.2%}")

# 4.3 PCA 2D plot
pca_df = pd.DataFrame(pca_result[:, :3], columns=['PC1', 'PC2', 'PC3'])
pca_df['exam_quantile'] = df['exam_quantile']

fig = px.scatter(pca_df, x='PC1', y='PC2', color='exam_quantile',
                 title='PCA of Student Habits',
                 color_discrete_sequence=px.colors.qualitative.Set1)
fig.show()

# 4.4 PCA 3D plot
fig = px.scatter_3d(pca_df, x='PC1', y='PC2', z='PC3', color='exam_quantile',
                    title='PCA 3D – Student Habits',
                    color_discrete_sequence=px.colors.qualitative.Set1)
fig.show()

# 4.5 PCA Loadings
loadings = pd.DataFrame(pca.components_.T, index=X.columns, 
                        columns=[f'PC{i+1}' for i in range(pca.components_.shape[0])])

print("\nTop 10 variables contributing to PC1:")
print(loadings['PC1'].abs().sort_values(ascending=False).head(10))

print("\nTop 10 variables contributing to PC2:")
print(loadings['PC2'].abs().sort_values(ascending=False).head(10))

# 4.6 Loadings heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(loadings.iloc[:, :4], annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('PCA Loadings (first 4 components)')
plt.tight_layout()
plt.show()

# =============================================================================
# 5. t-SNE (like WDBC2026_2.R)
# =============================================================================

# 5.1 Default t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=123)
tsne_result = tsne.fit_transform(X_scaled)

tsne_df = pd.DataFrame({
    'tSNE1': tsne_result[:, 0],
    'tSNE2': tsne_result[:, 1],
    'exam_quantile': df['exam_quantile']
})

fig = px.scatter(tsne_df, x='tSNE1', y='tSNE2', color='exam_quantile',
                 title='t-SNE of Student Habits (perplexity=30)',
                 color_discrete_sequence=px.colors.qualitative.Set1)
fig.show()

# 5.2 t-SNE hyperparameter exploration
perplexities = [10, 15, 20, 30]
etas = [10, 50, 100, 200]

fig, axes = plt.subplots(len(etas), len(perplexities), figsize=(15, 10))

for i, eta in enumerate(etas):
    for j, perp in enumerate(perplexities):
        tsne = TSNE(n_components=2, perplexity=perp, learning_rate=eta,
                    random_state=123, max_iter=500)
        result = tsne.fit_transform(X_scaled)
        
        axes[i, j].scatter(result[:, 0], result[:, 1], 
                          c=pd.Categorical(df['exam_quantile']).codes,
                          cmap='Set1', s=10, alpha=0.6)
        axes[i, j].set_title(f'perp={perp}, eta={eta}')
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.tight_layout()
plt.suptitle('t-SNE Hyperparameter Exploration', y=1.02, fontsize=16)
plt.show()

# 5.3 3D t-SNE
tsne_3d = TSNE(n_components=3, perplexity=30, learning_rate=200, random_state=123)
tsne_3d_result = tsne_3d.fit_transform(X_scaled)

tsne_3d_df = pd.DataFrame({
    'x': tsne_3d_result[:, 0],
    'y': tsne_3d_result[:, 1],
    'z': tsne_3d_result[:, 2],
    'exam_quantile': df['exam_quantile']
})

fig = px.scatter_3d(tsne_3d_df, x='x', y='y', z='z', color='exam_quantile',
                    title='t-SNE 3D – Student Habits',
                    color_discrete_sequence=px.colors.qualitative.Set1)
fig.show()

# =============================================================================
# 6. UMAP (like WDBC2026_3.R)
# =============================================================================

# 6.1 Default UMAP
umap_model = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=123)
umap_result = umap_model.fit_transform(X_scaled)

umap_df = pd.DataFrame({
    'UMAP1': umap_result[:, 0],
    'UMAP2': umap_result[:, 1],
    'exam_quantile': df['exam_quantile']
})

fig = px.scatter(umap_df, x='UMAP1', y='UMAP2', color='exam_quantile',
                 title='UMAP of Student Habits (n_neighbors=15, min_dist=0.1)',
                 color_discrete_sequence=px.colors.qualitative.Set1)
fig.show()

# 6.2 UMAP hyperparameter exploration
n_neighbors_list = [10, 20, 50, 100]
min_dists = [0.1, 0.5, 0.9]

fig, axes = plt.subplots(len(min_dists), len(n_neighbors_list), figsize=(15, 10))

for i, min_dist in enumerate(min_dists):
    for j, n_neigh in enumerate(n_neighbors_list):
        umap_model = umap.UMAP(n_components=2, n_neighbors=n_neigh,
                               min_dist=min_dist, random_state=123)
        result = umap_model.fit_transform(X_scaled)
        
        axes[i, j].scatter(result[:, 0], result[:, 1],
                          c=pd.Categorical(df['exam_quantile']).codes,
                          cmap='Set1', s=10, alpha=0.6)
        axes[i, j].set_title(f'n_neigh={n_neigh}, min_dist={min_dist}')
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.tight_layout()
plt.suptitle('UMAP Hyperparameter Exploration', y=1.02, fontsize=16)
plt.show()

# 6.3 3D UMAP
umap_3d = umap.UMAP(n_components=3, n_neighbors=50, min_dist=0.5, random_state=123)
umap_3d_result = umap_3d.fit_transform(X_scaled)

umap_3d_df = pd.DataFrame({
    'x': umap_3d_result[:, 0],
    'y': umap_3d_result[:, 1],
    'z': umap_3d_result[:, 2],
    'exam_quantile': df['exam_quantile']
})

fig = px.scatter_3d(umap_3d_df, x='x', y='y', z='z', color='exam_quantile',
                    title='UMAP 3D – Student Habits',
                    color_discrete_sequence=px.colors.qualitative.Set1)
fig.show()

# =============================================================================
# 7. CLUSTERING (like WDBC2026_4_1.R & WDBC2026_4_2.R)
# =============================================================================

# 7.1 K-Means: Determine optimal number of clusters
inertias = []
silhouettes = []
db_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))

# Plot all metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Method')

axes[2].plot(K_range, db_scores, 'go-')
axes[2].set_xlabel('k')
axes[2].set_ylabel('Davies-Bouldin Score')
axes[2].set_title('Davies-Bouldin Method')

plt.tight_layout()
plt.show()

# Choose optimal k (max silhouette)
optimal_k = K_range[np.argmax(silhouettes)]
print(f"\nOptimal number of clusters (K-Means): {optimal_k}")

# 7.2 Final K-Means model
kmeans_final = KMeans(n_clusters=optimal_k, random_state=123, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_scaled)
df['kmeans_cluster'] = kmeans_labels

# Cluster centroids
centroids = pd.DataFrame(kmeans_final.cluster_centers_, columns=X.columns)
print("\nCluster Centroids:")
print(centroids)

# Cluster summary
cluster_summary = df.groupby('kmeans_cluster')[numeric_cols].mean()
print("\nCluster Summary (mean values):")
print(cluster_summary.round(2))

# 7.3 K-Means with cross-validation
from sklearn.model_selection import StratifiedKFold

cv_scores = []
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

for train_idx, test_idx in kf.split(X_scaled, pd.Categorical(df['exam_quantile']).codes):
    X_train = X_scaled[train_idx]
    X_test = X_scaled[test_idx]
    
    kmeans_cv = KMeans(n_clusters=optimal_k, random_state=123, n_init=10)
    kmeans_cv.fit(X_train)
    labels_cv = kmeans_cv.predict(X_test)
    score = silhouette_score(X_test, labels_cv)
    cv_scores.append(score)

print(f"\nCross-validated Silhouette Score: {np.mean(cv_scores):.3f} (+/- {np.std(cv_scores):.3f})")

# =============================================================================
# 8. HIERARCHICAL CLUSTERING (like WDBC2026_4_1.R)
# =============================================================================

# 8.1 Multiple linkage methods
linkage_methods = ['average', 'single', 'complete', 'ward']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, method in enumerate(linkage_methods):
    Z = linkage(X_scaled, method=method)
    coph_dist = cophenet(Z, pdist(X_scaled))
    
    axes[idx].set_title(f"Linkage: {method}\nCophenetic Corr: {coph_dist[0]:.3f}")
    dendrogram(Z, ax=axes[idx], no_labels=True, color_threshold=0.7)
    axes[idx].axhline(y=0.7, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Best linkage method
best_method = max(linkage_methods, key=lambda x: cophenet(linkage(X_scaled, method=x), pdist(X_scaled))[0])
print(f"\nBest linkage method: {best_method}")

# 8.2 Ward method (usually best)
Z_ward = linkage(X_scaled, method='ward')

# Cut dendrogram at optimal_k
hier_labels = fcluster(Z_ward, t=optimal_k, criterion='maxclust')
df['hier_cluster'] = hier_labels

# Plot dendrogram with cut
plt.figure(figsize=(12, 6))
dendrogram(Z_ward, no_labels=True, color_threshold=0.7)
plt.axhline(y=0.7, color='red', linestyle='--', label=f'Cut at k={optimal_k}')
plt.title('Hierarchical Clustering Dendrogram (Ward)')
plt.legend()
plt.xlabel('Samples')
plt.ylabel('Distance')
plt.show()

# Compare with K-Means
ari = adjusted_rand_score(df['kmeans_cluster'], df['hier_cluster'])
nmi = normalized_mutual_info_score(df['kmeans_cluster'], df['hier_cluster'])
print(f"\nAgreement between K-Means and Hierarchical: ARI={ari:.3f}, NMI={nmi:.3f}")

# =============================================================================
# 9. DBSCAN & OPTICS (like IdealClustering.R)
# =============================================================================

# 9.1 DBSCAN - k-distance plot
k = 10
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_scaled)
distances, indices = nn.kneighbors(X_scaled)

k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 6))
plt.plot(k_distances)
plt.axhline(y=0.5, color='red', linestyle='--', label='eps=0.5')
plt.xlabel('Points sorted by distance')
plt.ylabel(f'Distance to {k}th nearest neighbor')
plt.title(f'k-distance plot (k={k})')
plt.legend()
plt.show()

# DBSCAN with different eps values
eps_values = [0.3, 0.5, 0.7, 1.0]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, eps in enumerate(eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=10)
    labels_db = dbscan.fit_predict(X_scaled)
    
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
    noise = np.sum(labels_db == -1)
    
    axes[idx].scatter(X_scaled[:, 0], X_scaled[:, 1], 
                     c=labels_db, cmap='Set1', s=30, alpha=0.6)
    axes[idx].set_title(f'DBSCAN eps={eps}, min_samples=10\nClusters: {n_clusters}, Noise: {noise}')
    axes[idx].set_xticks([])
    axes[idx].set_yticks([])

plt.tight_layout()
plt.show()

# Choose best DBSCAN (if any clusters found)
best_eps = None
best_n_clusters = 0
for eps in eps_values:
    dbscan = DBSCAN(eps=eps, min_samples=10)
    labels = dbscan.fit_predict(X_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters > best_n_clusters:
        best_n_clusters = n_clusters
        best_eps = eps

if best_eps is not None and best_n_clusters > 0:
    dbscan_best = DBSCAN(eps=best_eps, min_samples=10)
    dbscan_labels = dbscan_best.fit_predict(X_scaled)
    df['dbscan_cluster'] = dbscan_labels
    print(f"\nBest DBSCAN: eps={best_eps}, clusters={best_n_clusters}")

# 9.2 OPTICS
optics = OPTICS(min_samples=10, xi=0.05)
optics_labels = optics.fit_predict(X_scaled)

# Plot reachability
plt.figure(figsize=(12, 4))
plt.plot(optics.reachability_[optics.ordering_])
plt.xlabel('Ordering')
plt.ylabel('Reachability Distance')
plt.title('OPTICS Reachability Plot')
plt.show()

# Plot clusters
n_clusters_optics = len(set(optics_labels)) - (1 if -1 in optics_labels else 0)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], 
                     c=optics_labels, cmap='Set1', s=30, alpha=0.6)
plt.title(f'OPTICS Clustering\nClusters: {n_clusters_optics}')
plt.colorbar(scatter)
plt.show()

df['optics_cluster'] = optics_labels

# 9.3 HDBSCAN
try:
    import hdbscan
    
    hdb = hdbscan.HDBSCAN(min_cluster_size=10, gen_min_span_tree=True)
    hdb_labels = hdb.fit_predict(X_scaled)
    
    n_clusters_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
    
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], 
                         c=hdb_labels, cmap='Set1', s=30, alpha=0.6)
    plt.title(f'HDBSCAN Clustering\nClusters: {n_clusters_hdb}')
    plt.colorbar(scatter)
    plt.show()
    
    df['hdbscan_cluster'] = hdb_labels
    
except ImportError:
    print("\nHDBSCAN not installed. Install with: pip install hdbscan")
    hdb_labels = None

# =============================================================================
# 10. GAUSSIAN MIXTURE MODELS (like IdealClustering.R)
# =============================================================================

n_components_range = range(1, 11)
bic_scores = []
aic_scores = []

for n in n_components_range:
    gmm = GaussianMixture(n_components=n, random_state=123)
    gmm.fit(X_scaled)
    bic_scores.append(gmm.bic(X_scaled))
    aic_scores.append(gmm.aic(X_scaled))

plt.figure(figsize=(10, 6))
plt.plot(n_components_range, bic_scores, 'bo-', label='BIC')
plt.plot(n_components_range, aic_scores, 'ro-', label='AIC')
plt.xlabel('Number of Components')
plt.ylabel('Score')
plt.title('GMM Model Selection')
plt.legend()
plt.show()

optimal_gmm = n_components_range[np.argmin(bic_scores)]
print(f"\nOptimal number of GMM components: {optimal_gmm}")

# Final GMM model
gmm_final = GaussianMixture(n_components=optimal_gmm, random_state=123)
gmm_labels = gmm_final.fit_predict(X_scaled)
df['gmm_cluster'] = gmm_labels

print("\nGMM Weights:")
print(gmm_final.weights_)

# =============================================================================
# 11. CLUSTER VALIDATION & COMPARISON
# =============================================================================

# Collect all cluster labels
cluster_methods = {
    'K-Means': df['kmeans_cluster'],
    'Hierarchical': df['hier_cluster'],
    'GMM': df['gmm_cluster']
}

# Add DBSCAN if available
if 'dbscan_labels' in locals() and best_n_clusters > 0:
    cluster_methods['DBSCAN'] = dbscan_labels
if 'optics_labels' in locals() and n_clusters_optics > 0:
    cluster_methods['OPTICS'] = optics_labels
if 'hdb_labels' in locals() and hdb_labels is not None and n_clusters_hdb > 0:
    cluster_methods['HDBSCAN'] = hdb_labels

# Compute metrics for each method
metrics_df = pd.DataFrame()

for method, labels in cluster_methods.items():
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters > 1 and not np.all(labels == -1):
        sil = silhouette_score(X_scaled, labels)
        db = davies_bouldin_score(X_scaled, labels)
        ch = calinski_harabasz_score(X_scaled, labels)
    else:
        sil = np.nan
        db = np.nan
        ch = np.nan
    
    metrics_df[method] = [n_clusters, sil, db, ch]

metrics_df.index = ['n_clusters', 'Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']
print("\n=== Clustering Method Comparison ===")
print(metrics_df.round(3))

# Best method by Silhouette
if not metrics_df.loc['Silhouette'].isna().all():
    best_method = metrics_df.loc['Silhouette'].idxmax()
    print(f"\n🏆 Best method by Silhouette Score: {best_method}")
else:
    best_method = 'K-Means'
    print("\n⚠️ Could not determine best method, using K-Means as default.")

# Visualize best method
best_labels = df[cluster_methods[best_method].name]

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], 
                      c=best_labels, cmap='Set1', s=30, alpha=0.6)
plt.title(f'Best Clustering Method: {best_method}')
plt.colorbar(scatter)
plt.show()

# =============================================================================
# 12. COMPARE WITH EXAM QUANTILES
# =============================================================================

quantile_codes = pd.Categorical(df['exam_quantile']).codes

comparison_df = pd.DataFrame()
for method, labels in cluster_methods.items():
    if len(set(labels)) > 1 and not np.all(labels == -1):
        ari = adjusted_rand_score(quantile_codes, labels)
        nmi = normalized_mutual_info_score(quantile_codes, labels)
        comparison_df[method] = [ari, nmi]

comparison_df.index = ['ARI', 'NMI']
print("\n=== Agreement with Exam Quantiles ===")
print(comparison_df.round(3))

# Confusion matrix for best method
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(quantile_codes, best_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix: {best_method} vs Exam Quantiles')
plt.xlabel('Cluster')
plt.ylabel('Quantile')
plt.show()

# =============================================================================
# 13. FINAL VISUALIZATION: PCA with Clusters
# =============================================================================

# PCA projection with best clusters
pca_plot = pd.DataFrame(pca_result[:, :2], columns=['PC1', 'PC2'])
pca_plot['cluster'] = best_labels
pca_plot['exam_quantile'] = df['exam_quantile']

# Two views: by cluster and by quantile
fig = make_subplots(rows=1, cols=2, subplot_titles=['Clusters', 'Exam Quantiles'])

# Cluster view
fig.add_trace(
    go.Scatter(x=pca_plot['PC1'], y=pca_plot['PC2'], 
               mode='markers', 
               marker=dict(color=pca_plot['cluster'], 
                           colorscale='Set1', size=8),
               showlegend=False),
    row=1, col=1
)

# Quantile view
for quantile in pca_plot['exam_quantile'].unique():
    subset = pca_plot[pca_plot['exam_quantile'] == quantile]
    fig.add_trace(
        go.Scatter(x=subset['PC1'], y=subset['PC2'], 
                   mode='markers', marker=dict(size=8),
                   name=quantile),
        row=1, col=2
    )

fig.update_layout(height=500, width=1000, title_text="PCA Projection of Clusters")
fig.show()

# =============================================================================
# 14. FINAL SUMMARY
# =============================================================================

print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print(f"\nDataset: {len(df)} students with {len(numeric_cols)-1} habit features")
print(f"\nQuantile groups:")
for q in df['exam_quantile'].unique():
    print(f"  {q}: {len(df[df['exam_quantile'] == q])} students")

print(f"\nOptimal number of clusters (K-Means): {optimal_k}")
print(f"\nBest clustering method: {best_method}")

print("\nCluster profiles (mean values):")
print(cluster_summary.round(2))

print("\nCluster sizes:")
print(df.groupby('kmeans_cluster').size())

print("\n" + "="*60)
print("✅ Analysis complete! All methods from your R scripts have been replicated in Python.")
print("="*60)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load your CSV file
df = pd.read_csv('habits.csv')

# 2. Select features for clustering (numerical columns only)
# Option A: Use all numerical columns
features = df.select_dtypes(include=[np.number]).columns.tolist()

# Option B: Select specific columns manually
# features = ['column1', 'column2', 'column3']

# 3. Extract the feature matrix
X = df[features].copy()

# 4. Handle missing values (if any)
X = X.dropna()  # Or use: X.fillna(X.mean(), inplace=True)

# 5. Scale the features (important for k-means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Determine optimal number of clusters (Elbow Method)
inertia = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()

# 7. Choose number of clusters (e.g., from elbow method)
optimal_k = 3  # Adjust based on elbow plot

# 8. Apply K-Means
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# 9. View results
print("\nCluster Centers (scaled):")
print(kmeans.cluster_centers_)

print("\nCluster Distribution:")
print(df['Cluster'].value_counts())

print("\nFirst few rows with cluster assignments:")
print(df.head())

# 10. Save results
df.to_csv('clustered_data.csv', index=False)

# 11. Visualize clusters (for 2D data)
if len(features) >= 2:
    # Option A: Use first two features
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], 
                         c=df['Cluster'], cmap='viridis', alpha=0.6)
    plt.xlabel(features[0])
    plt.ylabel(features[1])
    plt.title(f'K-Means Clustering (k={optimal_k})')
    plt.colorbar(scatter)
    plt.show()